# 00 — What can this dataset actually say about radiation?

Before any RT contrast is run, this notebook establishes what the two objects are *able* to
answer, because the honest answer is "different things", and getting it backwards wastes a
month. It produces no biology. It produces the design tables, the census, and the
minimum-detectable-effect table that every later notebook reads.

Four questions, each of which changes what `01`-`04` are allowed to attempt:

| § | Question | Why it gates everything |
|---|---|---|
| 2-3 | **Single cell: how many cells of each type, per sample?** | If the lymphoid compartment is ~18 cells a sample, no amount of clever statistics makes it a T-cell dataset. |
| 4 | **Can CAR-T cells be identified by their transgene?** | `PSCA` and `TAG72` are in `var_names`. If they work, the whole project reorients around the transferred cells. |
| 5 | **Spatial: how variable is tissue content between sections?** | Decides whether spatial can contribute *fractions* or only *within-tissue geometry*. |
| 6 | **What exactly is the hemisphere split?** | It is being counted on for abscopal power. This checks whether it delivers that, or something else. |
| 7 | **What effect size is detectable, per cell type?** | So that a null in `01` can be reported as "absent" rather than "we could not have seen it". |

Nothing here is assumed from the `spp1Analysis` README — every number is recomputed, because two
of that project's real costs came from an assumed fact (the `CyT72`/`CyTAG72` arm-name mismatch,
and `.X` being read as counts).

## 1. Config

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import re, time
from pathlib import Path

import numpy as np, pandas as pd, h5py
import anndata as ad
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.width", 260); pd.set_option("display.max_columns", 60)

# ---- paths: HPC first, local copy as fallback (same pattern as 08/09) -------------------
_HPC = Path("/coh_labs/yunroseli/Jona/CAR-T")
_LOC = Path("/Users/janzules/Roselab/Spatial/CAR_T")
PROJ = _HPC if _HPC.exists() else _LOC
ON_HPC = PROJ is _HPC

SC_REF    = (PROJ/"data/zarr/fullDataset/annotating_references/annotation_v4.h5ad" if ON_HPC
             else PROJ/"data/sc-reference/annotation_v4.h5ad")
ZARR_FILE = (PROJ/"data/zarr/fullDataset/processing_Zarr/1_C2l_annotated_CC" if ON_HPC
             else PROJ/"data/zarrFiles/1_C2l_annotations_400")
HEMI_DIR  = PROJ/"docs"

OUT = PROJ/"results/rt_advantage"; FIG = OUT/"figures"
OUT.mkdir(parents=True, exist_ok=True); FIG.mkdir(parents=True, exist_ok=True)

# The hemisphere barcode file originates on the Mac; if the kernel is on the HPC it has to be
# copied across. .csv is accepted as well as .xlsx so the section also runs without openpyxl.
HEMI_FILE = next((p for p in [HEMI_DIR/"B_subsample_barcodes.xlsx",
                              HEMI_DIR/"B_subsample_barcodes.csv"] if p.exists()), None)

for p, lbl in [(SC_REF, "single cell"), (ZARR_FILE, "spatial"), (HEMI_FILE, "hemispheres")]:
    print(f"{lbl:<12s} {'OK      ' if p is not None and p.exists() else 'MISSING '} {p}")
if HEMI_FILE is None:
    print(f"  -> section 6 will be skipped. Copy B_subsample_barcodes.xlsx (or .csv) to "
          f"{HEMI_DIR}/")

# ---- the arm-name mapping, written once so nothing downstream re-types it ---------------
# Spatial spells the non-specific CAR-T arm CyT72/RTCyT72; single cell spells it
# CyTAG72/RTCyTAG72. Mismatching silently drops every CAR-T-background contrast.
ARMS = pd.DataFrame([
    # canonical    spatial      single-cell   RT     cart            cy
    ("NoTx",       "NoTx",      "NoTx",       False, "none",         False),
    ("CyPSCA",     "CyPSCA",    "CyPSCA",     False, "PSCA",         True ),
    ("CyT72",      "CyT72",     "CyTAG72",    False, "nonspecific",  True ),
    ("RTCyPSCA",   "RTCyPSCA",  "RTCyPSCA",   True,  "PSCA",         True ),
    ("RTCyT72",    "RTCyT72",   "RTCyTAG72",  True,  "nonspecific",  True ),
], columns=["arm", "spatial_name", "sc_name", "RT", "cart", "cy"])

# the RT contrast, run once per CAR-T background; sign-agreement across the two is the evidence
CART_PAIRS = [("PSCA", "RTCyPSCA", "CyPSCA"), ("nonspecific", "RTCyT72", "CyT72")]
SITE = {1: "irradiated", 2: "abscopal"}      # loc 1 / Tu1 = in beam, loc 2 / Tu2 = out of beam

MICRONS_PER_PIXEL = 0.34
SEED = 0

ARMS

## 2. Single-cell design

`file_name` is `ARM_Tu{site}_{rep}`. `rep` indexes the animal within an arm, so `Tu1` and `Tu2`
of the same `rep` are **the same mouse** — that pairing is what makes contrast 3 (the
difference-in-differences) possible, and it is the only within-animal handle in the study.

In [ ]:
with h5py.File(SC_REF, "r") as f:
    sc_obs = ad.io.read_elem(f["obs"])
    sc_var = ad.io.read_elem(f["var"])
print(f"{len(sc_obs):,} cells x {len(sc_var):,} genes")

o = sc_obs
o["sample"] = o["file_name"].astype(str)
parts       = o["sample"].str.extract(r"^(?P<sc_name>.+)_Tu(?P<site>\d)_(?P<rep>\d+)$")
assert parts.notna().all().all(), "file_name did not parse — check the pattern before going on"
o["sc_name"] = parts["sc_name"]
o["site"]    = parts["site"].astype(int)
o["rep"]     = parts["rep"].astype(int)
o["mouse"]   = o["sc_name"] + "_m" + parts["rep"]        # same mouse across Tu1/Tu2
o["arm"]     = o["sc_name"].map(dict(zip(ARMS.sc_name, ARMS.arm)))
assert o["arm"].notna().all(), f"unmapped arm names: {set(o.loc[o.arm.isna(),'sc_name'])}"
o["ct"]      = o["cell_type_lvl2"].astype(str)

sc_design = (o.groupby("sample", observed=True)
               .agg(arm=("arm","first"), site=("site","first"), rep=("rep","first"),
                    mouse=("mouse","first"), n_cells=("arm","size"))
               .reset_index())
sc_design["site_label"] = sc_design["site"].map(SITE)
sc_design = sc_design.merge(ARMS[["arm","RT","cart","cy"]], on="arm")
sc_design.to_csv(OUT/"design_singlecell.csv", index=False)

print("\nsamples per arm x site (mice):")
print(sc_design.pivot_table(index="arm", columns="site_label", values="sample", aggfunc="count")
        .reindex(ARMS.arm).fillna(0).astype(int).to_string())
print("\ncells per arm x site:")
print(sc_design.pivot_table(index="arm", columns="site_label", values="n_cells", aggfunc="sum")
        .reindex(ARMS.arm).fillna(0).astype(int).to_string())

paired = sc_design.groupby("mouse")["site"].nunique()
print(f"\nmice with BOTH Tu1 and Tu2 (usable for the within-animal DiD): "
      f"{int((paired==2).sum())}/{len(paired)}")
if (paired < 2).any():
    print(f"  unpaired: {sorted(paired[paired<2].index)}")

## 3. Single-cell compartment census

The number that decides the shape of this project. Read the `CD8_T` column and the `lymphoid`
row before planning anything about the effector arm.

In [ ]:
LYMPHOID = ["CD8_T", "CD4_T", "Treg", "NK", "NKT", "B"]
MAC      = ["M1_like_Mac", "M2_like_Mac", "Intermediate_Mac"]
MONO     = ["Classical_Mono", "Nonclassical_Mono"]
NEU      = ["N1_like_Neu", "N2_like_Neu"]

sc_census = pd.crosstab(o["sample"], o["ct"])
sc_census = sc_census.reindex(sc_design.set_index("sample").index)
sc_census.to_csv(OUT/"census_singlecell_per_sample.csv")

tot = sc_census.sum(0).sort_values(ascending=False)
summary = pd.DataFrame({
    "total_cells":   tot,
    "pct_of_object": 100*tot/tot.sum(),
    "median_per_sample": sc_census.median(0).reindex(tot.index),
    "min_per_sample":    sc_census.min(0).reindex(tot.index),
    "max_per_sample":    sc_census.max(0).reindex(tot.index),
    "samples_with_>=20": (sc_census >= 20).sum(0).reindex(tot.index),
})
print("SINGLE CELL — per cell type (29 samples)")
print(summary.round(2).to_string())

grp = {"lymphoid": LYMPHOID, "macrophage": MAC, "monocyte": MONO, "neutrophil": NEU,
       "tumour": ["Cancer_cell"], "stroma": ["Endothelial","Fibroblast"]}
print("\ncompartment totals:")
for k, v in grp.items():
    cols = [c for c in v if c in sc_census.columns]
    n = int(sc_census[cols].sum().sum())
    per = sc_census[cols].sum(1)
    print(f"  {k:<11s} {n:>7,} cells ({100*n/tot.sum():4.1f}%)   per sample: "
          f"median {per.median():.0f}, range {per.min():.0f}-{per.max():.0f}")

## 4. Can CAR-T cells be found by their transgene?

`var_names` carries `PSCA` and `TAG72` alongside mouse `Psca` — capitalised, and with no
`chromosome` entry, which is what a spiked-in construct reference looks like. If they carry
signal, the transferred cells become directly addressable and this project changes shape.

Three things are checked, and the third is the one that matters:
1. are they really non-genomic features (no chromosome)?
2. **arm specificity** — `PSCA` reads must appear only in PSCA arms, `TAG72` only in TAG72 arms,
   and neither in `NoTx`. This is a free, perfect negative control.
3. **how many cells** carry them.

The counts layer is CSR and 255M nonzeros, so it is scanned in chunks for the two columns rather
than sliced (column slicing a CSR of this size materialises the whole thing).

In [ ]:
gene_index = list(sc_var.index.astype(str))
TG = {}
for name in ["PSCA", "TAG72"]:
    if name in gene_index:
        i = gene_index.index(name)
        TG[i] = name
        chrom = sc_var["chromosome"].iloc[i] if "chromosome" in sc_var else "?"
        print(f"{name:<6s} var index {i:>6d}   chromosome = {chrom}   "
              f"({'non-genomic -> construct reference' if pd.isna(chrom) else 'GENOMIC — not a construct!'})")
    else:
        print(f"{name}: absent from var_names")
# a genomic control gene so the scan is shown to work at all
# genomic controls: Ptprc shows the scan works, Psca IS the PSCA-CAR target antigen
for name in ["Cd8a", "Ptprc", "Psca"]:
    if name in gene_index: TG[gene_index.index(name)] = name

t0 = time.time()
with h5py.File(SC_REF, "r") as f:
    grp_c = f["layers/counts"] if "layers" in f and "counts" in f["layers"] else f["X"]
    indptr = grp_c["indptr"][:]
    ind, dat = grp_c["indices"], grp_c["data"]
    N, CH = ind.shape[0], 20_000_000
    acc = {k: np.zeros(len(o)) for k in TG}
    want = np.array(sorted(TG))
    for s in range(0, N, CH):
        e = min(s + CH, N)
        ic = ind[s:e]
        m = np.isin(ic, want)
        if not m.any(): continue
        pos = np.nonzero(m)[0]
        rows = np.searchsorted(indptr, s + pos, side="right") - 1
        vals, cols = dat[s:e][pos], ic[pos]
        for k in TG:
            sel = cols == k
            if sel.any(): np.add.at(acc[k], rows[sel], vals[sel])
print(f"\nscanned {N:,} nonzeros in {time.time()-t0:.0f}s")

for k, name in TG.items(): o[f"tg_{name}"] = acc[k]
tg_cols = [f"tg_{n}" for n in TG.values()]

print("\ncells positive, and total UMI:")
for c in tg_cols:
    print(f"  {c:<10s} {int((o[c]>0).sum()):>6,} cells   {o[c].sum():>8,.0f} UMI")

print("\n--- SPECIFICITY: transgene+ cells by arm (the free negative control) ---")
spec = o.groupby("arm", observed=True).apply(
    lambda d: pd.Series({c: int((d[c] > 0).sum()) for c in tg_cols} | {"n_cells": len(d)}))
print(spec.reindex(ARMS.arm).to_string())

print("\n--- transgene+ cells by cell type ---")
pos = o[(o["tg_PSCA"] > 0) | (o["tg_TAG72"] > 0)] if {"tg_PSCA","tg_TAG72"} <= set(o) else o.iloc[:0]
print(pos["ct"].value_counts().head(10).to_string() if len(pos) else "  none")

n_tg = int(len(pos))
CAR_USABLE = n_tg >= 200
print(f"\nVERDICT: {n_tg} transgene-positive cells -> "
      + ("usable, reorient the project around them." if CAR_USABLE else
         "NOT usable. Transferred cells must be analysed as part of the CD8 compartment."))

# --- is the CAR target antigen itself measurable? ---
TARGET_USABLE = False
if "tg_Psca" in o:
    n_psca = int((o["tg_Psca"] > 0).sum())
    in_tum = int(((o["tg_Psca"] > 0) & (o["ct"] == "Cancer_cell")).sum())
    n_tum  = int((o["ct"] == "Cancer_cell").sum())
    TARGET_USABLE = in_tum >= 0.10*n_tum
    print("\n--- mouse Psca, the PSCA-CAR target antigen ---")
    print(f"  {n_psca:,} cells positive, {o['tg_Psca'].sum():,.0f} UMI total")
    print(f"  in Cancer_cell: {in_tum}/{n_tum:,} ({100*in_tum/max(n_tum,1):.1f}% of tumour cells)")
    print(f"  -> 'does RT raise the target antigen?' is "
          + ("ANSWERABLE here." if TARGET_USABLE else
             "NOT answerable here — the target is below detection in this assay."))
# save the positive cells and a per-sample tally, not all 93k rows — the answer is 8 cells
if len(pos):
    pos[["sample", "arm", "site", "mouse", "ct"] + tg_cols].to_csv(
        OUT/"transgene_positive_cells.csv", index=False)
(o.groupby(["sample", "arm", "site"], observed=True)[tg_cols]
   .agg(lambda v: int((v > 0).sum())).reset_index()
   .to_csv(OUT/"transgene_audit_per_sample.csv", index=False))

## 5. Spatial design and census — and the dynamic-range problem

The spatial object has ~90x the lymphoid cells of the single-cell object, which is why the
effector arm is a spatial question. But sections differ enormously in what tissue they caught,
and that shows up as orders-of-magnitude swings in per-tissue counts. The quantity printed at
the end — the ratio of the 90th to the 10th percentile of each cell type's *fraction* across
tissues — is what decides whether spatial fractions can carry an RT contrast at n = 2-4.

In [ ]:
import spatialdata as spd
sdata  = spd.read_zarr(ZARR_FILE)
sp_ad  = sdata.tables["segmentation_counts"]
print(f"{sp_ad.n_obs:,} cells x {sp_ad.n_vars:,} genes")

so = sp_ad.obs
so["tissue"]   = so["tissue"].astype(str)
so["arm"]      = so["condition"].astype(str).map(dict(zip(ARMS.spatial_name, ARMS.arm)))
assert so["arm"].notna().all(), f"unmapped: {set(so.loc[so.arm.isna(),'condition'].astype(str))}"
so["site"]     = so["tumor_loc"].astype(int)
so["ct"]       = so["c2l_consolidated"].astype(str)

sp_design = (so.groupby("tissue", observed=True)
               .agg(arm=("arm","first"), site=("site","first"), mouse=("mouse","first"),
                    n_cells=("arm","size")).reset_index())
sp_design["rep"] = sp_design["tissue"].str.extract(r"_(\d+)$").astype(int)
sp_design["site_label"] = sp_design["site"].map(SITE)
sp_design = sp_design.merge(ARMS[["arm","RT","cart","cy"]], on="arm")
sp_design.to_csv(OUT/"design_spatial.csv", index=False)

print("\ntissues per arm x site:")
print(sp_design.pivot_table(index="arm", columns="site_label", values="tissue", aggfunc="count")
        .reindex(ARMS.arm).fillna(0).astype(int).to_string())

sp_census = pd.crosstab(so["tissue"], so["ct"])
sp_census.to_csv(OUT/"census_spatial_per_tissue.csv")
tot_s = sp_census.sum(0).sort_values(ascending=False)

frac = sp_census.div(sp_census.sum(1), axis=0)
rng  = pd.DataFrame({
    "total_cells": tot_s,
    "median_per_tissue": sp_census.median(0).reindex(tot_s.index),
    "min_per_tissue": sp_census.min(0).reindex(tot_s.index),
    "max_per_tissue": sp_census.max(0).reindex(tot_s.index),
    "frac_p10": frac.quantile(0.10).reindex(tot_s.index),
    "frac_p90": frac.quantile(0.90).reindex(tot_s.index),
})
rng["p90_over_p10"] = rng["frac_p90"] / rng["frac_p10"].replace(0, np.nan)
print("\nSPATIAL — per cell type across 32 tissues")
print(rng.round(4).to_string())

bad = rng[rng["p90_over_p10"] > 10].index.tolist()
print(f"\ncell types whose tissue-level FRACTION swings >10x between the 10th and 90th "
      f"percentile: {len(bad)}/{len(rng)}")
print(f"  {bad}")
print("\n-> any of the above cannot carry a fraction-based RT contrast at n = 2-4 tissues a "
      "side.\n   Spatial contributes WITHIN-tissue geometry; fractions come from single cell.")

## 6. The hemisphere split — what it actually is

`docs/B_subsample_barcodes.xlsx` splits the two `RTCyPSCA` abscopal sections in half, to get more
out of an n = 2 arm. This section establishes three things, in order of how much they change the
conclusion that can be drawn from it:

1. **do the barcodes join?** (they must match spatial `cell_id` exactly)
2. **what fraction of each section does the split cover, and is the covered part representative?**
3. **are the two halves exchangeable?** — because that is the assumption under "n = 2 becomes n = 4".

The honest answer to (3) decides whether the split is a power device or a control device.

In [ ]:
RUN_HEMI = HEMI_FILE is not None
HT, cov, ex = [], pd.DataFrame(), pd.DataFrame()
so["hemi"] = "not_in_split"


def audit_hemispheres():
    """Returns (tissues, coverage_table, exchangeability_table). Prints the whole audit."""
    try:
        hemi = (pd.read_csv(HEMI_FILE) if HEMI_FILE.suffix == ".csv"
                else pd.read_excel(HEMI_FILE))
    except ImportError as e:      # openpyxl missing in this kernel
        raise ImportError(f"{e}\nEither `pip install openpyxl`, or save the sheet as "
                          f"{HEMI_DIR}/B_subsample_barcodes.csv and re-run.") from None
    print(f"{len(hemi):,} barcodes | tissues: {sorted(hemi.tissue.unique())}")
    print(hemi.groupby(["tissue", "B_subsample"]).size().to_string())

    # key on (tissue, cell_id), not cell_id alone: segmentation ids are only unique within a
    # capture area, and a bare-barcode join silently labels cells in the wrong section.
    cellid = (so["cell_id"].astype(str) if "cell_id" in so.columns
              else pd.Series(so.index.astype(str), index=so.index))
    key = pd.MultiIndex.from_arrays([so["tissue"].values, cellid.values])
    lut = pd.Series(hemi.B_subsample.values,
                    index=pd.MultiIndex.from_arrays([hemi.tissue.values, hemi.barcode.values]))
    dup = int(lut.index.duplicated().sum())
    so["hemi"] = pd.Series(lut.reindex(key).values, index=so.index).fillna("not_in_split")

    print("\n--- (1) join integrity ---")
    matched = int((so["hemi"] != "not_in_split").sum())
    bare = int(cellid.isin(set(hemi.barcode)).sum())
    print(f"  {matched:,}/{len(hemi):,} barcodes matched on (tissue, cell_id) "
          f"({100*matched/len(hemi):.1f}%)   duplicate keys in file: {dup}")
    if bare != matched:
        print(f"  NOTE: a bare cell_id join would have matched {bare:,} cells "
              f"({bare-matched:+,}) — cell_id is reused across sections, so the tissue key "
              f"is required.")
    assert matched > 0.99*len(hemi), "barcodes do not join — check the cell_id convention"

    tissues = sorted(hemi.tissue.unique())

    print("\n--- (2) coverage, and is the covered region representative of the section? ---")
    cov_rows = []
    for t in tissues:
        s_ = so[so.tissue == t]
        ins = s_[s_.hemi != "not_in_split"]
        cov_rows.append(dict(tissue=t, tissue_cells=len(s_), split_cells=len(ins),
                             pct_covered=100*len(ins)/len(s_)))
        a = ins["ct"].value_counts(normalize=True)*100
        w = s_["ct"].value_counts(normalize=True)*100
        cmp = pd.DataFrame({"in_split_%": a, "whole_tissue_%": w}).fillna(0)
        cmp["abs_diff"] = (cmp["in_split_%"] - cmp["whole_tissue_%"]).abs()
        print(f"\n  {t}: {len(ins):,}/{len(s_):,} cells covered ({100*len(ins)/len(s_):.0f}%)")
        print("    " + cmp.sort_values("abs_diff", ascending=False).head(6).round(1)
                          .to_string().replace("\n", "\n    "))

    print("\n--- (2b) geometry: which axis is the cut, and is it clean? ---")
    xy = np.asarray(sp_ad.obsm["spatial"], float)
    so["_x"], so["_y"] = xy[:, 0], xy[:, 1]
    for t in tissues:
        s_ = so[(so.tissue == t) & (so.hemi != "not_in_split")]
        for ax in ["_x", "_y"]:
            hi = s_.loc[s_.hemi.str.endswith("_hi"), ax]
            lo = s_.loc[s_.hemi.str.endswith("_lo"), ax]
            span = max(hi.max(), lo.max()) - min(hi.min(), lo.min())
            ov = (min(hi.max(), lo.max()) - max(hi.min(), lo.min())) / span
            tag = "   <- clean partition, this is the cut axis" if ov <= 0.01 else ""
            print(f"  {t} {ax[1]}: hi [{hi.min():.0f},{hi.max():.0f}]  "
                  f"lo [{lo.min():.0f},{lo.max():.0f}]  range-overlap {ov:+.2f}{tag}")

    print("\n--- (3) are the two halves exchangeable? ---")
    ex_rows = []
    for t in tissues:
        s_ = so[(so.tissue == t) & (so.hemi != "not_in_split")]
        p = pd.crosstab(s_.hemi, s_.ct, normalize="index")*100
        d = (p.loc[f"{t}_hi"] - p.loc[f"{t}_lo"]).abs().sort_values(ascending=False)
        print(f"\n  {t} — largest |hi - lo| composition gaps:")
        for ctp in d.head(5).index:
            print(f"    {ctp:<18s} hi {p.loc[f'{t}_hi', ctp]:5.1f}%   "
                  f"lo {p.loc[f'{t}_lo', ctp]:5.1f}%   diff {d[ctp]:5.1f} pp")
        ex_rows.append(dict(tissue=t, max_gap_pp=float(d.max()), mean_gap_pp=float(d.mean()),
                            total_variation_pp=float(d.sum()/2)))
    return tissues, pd.DataFrame(cov_rows), pd.DataFrame(ex_rows)


if RUN_HEMI:
    HT, cov, ex = audit_hemispheres()
    audit = cov.merge(ex, on="tissue")
    audit.to_csv(OUT/"hemisphere_split_audit.csv", index=False)
    print("\n" + audit.round(1).to_string(index=False))
    print("\nreading: a split that covered the whole section and produced exchangeable halves "
          "would show\n  pct_covered ~100 and max_gap_pp of a few points. Anything else means "
          "the halves are two\n  different regions of one mouse, which is a reproducibility "
          "control, not extra n.")
else:
    print("SKIPPED — no hemisphere barcode file found under", HEMI_DIR)
    print("Everything else in this notebook stands; copy the file across and re-run this section.")

In [ ]:
if RUN_HEMI:
    fig, axes = plt.subplots(1, len(HT), figsize=(6.2*len(HT), 5.6))
    axes = np.atleast_1d(axes)
    for ax, t in zip(axes, HT):
        s_ = so[so.tissue == t]
        for g, c, lbl in [("not_in_split", "#d9d9d9", "not in split"),
                          (f"{t}_lo", "#3b6ea5", "lo"), (f"{t}_hi", "#d1495b", "hi")]:
            d = s_[s_.hemi == g]
            ax.scatter(d["_x"], d["_y"], s=0.4, c=c, lw=0, rasterized=True,
                       label=f"{lbl} (n={len(d):,})")
        ax.set_title(f"{t}\n{100*(s_.hemi != 'not_in_split').mean():.0f}% of the section is in "
                     f"the split", fontsize=11)
        ax.set_aspect("equal"); ax.invert_yaxis()
        ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
        ax.legend(markerscale=14, fontsize=9, loc="best", frameon=True)
    fig.suptitle("Hemisphere split — what it covers, and where the cut falls", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG/"hemisphere_split_geometry.png", dpi=250, bbox_inches="tight")
    plt.show()

## 7. Minimum detectable effect, per cell type

A null is only worth reporting next to the effect size that would have been caught. For each
cell type this computes, from the **observed within-arm spread of the CLR-transformed
composition**:

* `sd_within` — the actual biological + technical spread inside an arm x site cell;
* `mde80` — the smallest RT delta a two-sided t-test at n per side would call at 80% power.
  Reported for context only; at n = 3 this project's rule is sign-agreement, not p-values;
* `counting_floor` — the irreducible spread from counting statistics alone (`1/sqrt(k)` on the
  log scale, `k` = median cells per sample). **If `sd_within` is close to this, the cell type is
  noise-dominated and nothing will ever be detected in it**;
* `benchmark_treated_vs_NoTx` — the same statistic for a perturbation already known to be real.
  This is the yardstick: an RT effect is worth looking for only where it could plausibly be some
  fraction of this.

CLR (centered log-ratio) rather than raw proportion, because proportions are closed — a myeloid
influx deflates every other fraction arithmetically, and that artifact would otherwise be read
as a real depletion.

In [ ]:
PSEUDO = 0.5          # pseudocount for zero cells
MIN_MEDIAN = 5        # a cell type needs this many cells in the median sample to enter the basis

def clr(counts_df, pseudo=PSEUDO):
    """Centered log-ratio of a samples x celltypes count table."""
    x = counts_df.values.astype(float) + pseudo
    p = x / x.sum(1, keepdims=True)
    lg = np.log(p)
    return pd.DataFrame(lg - lg.mean(1, keepdims=True),
                        index=counts_df.index, columns=counts_df.columns)

basis = sc_census.columns[sc_census.median(0) >= MIN_MEDIAN].tolist()
print(f"CLR basis: {len(basis)}/{sc_census.shape[1]} cell types with median >= {MIN_MEDIAN} "
      f"cells/sample\n  in : {basis}\n  out: {[c for c in sc_census.columns if c not in basis]}")

Z = clr(sc_census[basis]).join(sc_design.set_index("sample")[["arm","site","RT","cart","mouse"]])

# within-arm-x-site sd, pooled across the four treated arms (NoTx excluded: it is a different
# tissue state, and pooling it would inflate the spread)
treated = Z[Z.arm != "NoTx"]
sd_within = (treated.groupby(["arm","site"], observed=True)[basis].std()
                    .pow(2).mean().pow(0.5))

n_side = 3
df_t = 2*n_side - 2
mde80 = (stats.t.ppf(0.975, df_t) + stats.t.ppf(0.80, df_t)) * sd_within * np.sqrt(2/n_side)

med_k = sc_census[basis].median(0)
counting_floor = 1/np.sqrt(med_k.clip(lower=1))

bench = (Z[Z.arm != "NoTx"][basis].mean() - Z[Z.arm == "NoTx"][basis].mean())

mde = pd.DataFrame({
    "median_cells_per_sample": med_k,
    "sd_within": sd_within,
    "counting_floor": counting_floor,
    "noise_dominated": (sd_within < 1.5*counting_floor),
    "mde80_n3": mde80,
    "benchmark_treated_vs_NoTx": bench,
    "mde_as_frac_of_benchmark": mde80/bench.abs(),
}).sort_values("mde_as_frac_of_benchmark")
mde.to_csv(OUT/"mde_singlecell_composition.csv")
print("\nSINGLE-CELL COMPOSITION — what is detectable (CLR units, log scale)")
print(mde.round(3).to_string())

good = mde.index[(mde.mde_as_frac_of_benchmark < 1) & (~mde.noise_dominated)].tolist()
print(f"\ncell types where an RT effect smaller than the known treated-vs-NoTx effect would be "
      f"detected:\n  {good}")
print(f"\nnoise-dominated (counting statistics, not biology): "
      f"{mde.index[mde.noise_dominated].tolist()}")

## 8. Summary

In [ ]:
print("="*94)
print("SUMMARY — report these back")
print("="*94)

print(f"\n[design]")
print(f"  single cell : {len(sc_design)} samples, {sc_design.n_cells.sum():,} cells, "
      f"{sc_design.mouse.nunique()} mice")
print(f"  spatial     : {len(sp_design)} tissues, {sp_design.n_cells.sum():,} cells")
print(f"  mice with both sites (DiD-usable): {int((sc_design.groupby('mouse')['site'].nunique()==2).sum())}")
print(f"  abscopal tissues: RT {sorted(sp_design.query('RT & site==2').tissue)}")
print(f"                    non-RT baseline {sorted(sp_design.query('~RT & cy & site==2').tissue)}")

print(f"\n[modality split]")
ly_sc = int(sc_census[[c for c in LYMPHOID if c in sc_census]].sum().sum())
ly_sp = int(sp_census[[c for c in LYMPHOID if c in sp_census]].sum().sum())
print(f"  lymphoid cells   single cell {ly_sc:,} ({100*ly_sc/len(o):.1f}%)  |  spatial {ly_sp:,}")
print(f"  CD8_T            single cell {int(sc_census.get('CD8_T', pd.Series(0)).sum()):,} "
      f"(median {sc_census.get('CD8_T', pd.Series([0])).median():.0f}/sample)  |  "
      f"spatial {int(sp_census.get('CD8_T', pd.Series(0)).sum()):,}")
print(f"  -> effector arm is {'SPATIAL-ONLY' if ly_sc < 5000 else 'readable in both'}")
print(f"  spatial cell types with >10x tissue-level fraction swing: {len(bad)}/{len(rng)}"
      f"  -> spatial fractions {'UNUSABLE' if len(bad) > len(rng)/3 else 'usable'} for RT contrasts")

print(f"\n[CAR transgene and target antigen]")
print(f"  transgene-positive cells: {n_tg}   -> {'USABLE' if CAR_USABLE else 'NOT USABLE'}")
print(f"  mouse Psca in tumour cells -> target-antigen question "
      f"{'ANSWERABLE' if TARGET_USABLE else 'NOT answerable'}")
if len(pos):
    print(f"  cell types carrying them: {dict(pos['ct'].value_counts().head(4))}")
    off = spec.loc[[a for a in ['NoTx'] if a in spec.index], [c for c in tg_cols if 'PSCA' in c or 'TAG72' in c]]
    print(f"  NoTx false positives (should be 0): {off.values.ravel().tolist()}")

print(f"\n[hemisphere split]")
if RUN_HEMI:
    for _, r in cov.merge(ex, on="tissue").iterrows():
        print(f"  {r.tissue}: covers {r.pct_covered:.0f}% of the section; "
              f"largest hi-vs-lo composition gap {r.max_gap_pp:.1f} pp "
              f"(total variation {r.total_variation_pp:.1f} pp)")
    _exch = ("NOT exchangeable -> use as a within-tissue reproducibility control, not as extra n"
             if ex.max_gap_pp.max() > 5 else "plausibly exchangeable -> may be pooled")
    print(f"  -> halves are {_exch}")
else:
    print("  not audited — barcode file not reachable from this kernel")

print(f"\n[detectability, single-cell composition]")
print(f"  usable readouts  : {good}")
print(f"  noise-dominated  : {mde.index[mde.noise_dominated].tolist()}")
print(f"  dropped from basis: {[c for c in sc_census.columns if c not in basis]}")

print(f"\n[files written to {OUT}]")
for f_ in sorted(OUT.glob('*.csv')): print(f"  {f_.name}")
print("="*94)